# Step 10: Daily digest

![Step 10 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/10-step.png)

## What you're building today

The final piece. A scheduled job that, every morning, summarizes yesterday's sightings and posts them to Slack.

Something like:

```
Yesterday at the feeder (5 sightings):
  - American Robin (×2)
  - Northern Cardinal (×1)
  - Blue Jay (×1)
  - House Sparrow (×1)
```

By the end, every box is wired and the bird watcher is complete. 🎉

## Step 10.1 — What is a scheduler?

A **scheduler** runs code at specific times. "Every morning at 9am", "every hour", "every minute".

There are lots of ways to do this:

- **`schedule`** (Python library) — easy to use, runs in your script. Good for laptops.
- **cron** (Unix) — built into Linux/macOS. Set it and forget it. Good for servers.
- **GitHub Actions** — schedule jobs from a repo. Free for public repos.
- **APScheduler** — heavier Python library, supports persistent jobs.

For the tutorial we'll use `schedule` because it runs inside this notebook — you can see it work in seconds.

In [ ]:
%pip install -q schedule
print("OK")

## Step 10.2 — Query yesterday's sightings

We need a SQL query that gets everything from the last 24 hours, grouped by species.

In [ ]:
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

DB_PATH = Path("data/birds.db")

def yesterday_sightings() -> list[dict]:
    """Get all sightings from the last 24 hours, grouped by species."""
    cutoff = (datetime.now() - timedelta(hours=24)).isoformat(timespec="seconds")
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute("""
            SELECT species, COUNT(*) AS n,
                   MAX(confidence) AS best_conf,
                   MAX(timestamp) AS latest
            FROM sightings
            WHERE timestamp >= ?
            GROUP BY species
            ORDER BY n DESC
        """, (cutoff,)).fetchall()
        return [dict(r) for r in rows]

sightings = yesterday_sightings()
total = sum(s["n"] for s in sightings)
print(f"Last 24 hours: {total} sightings, {len(sightings)} unique species")
for s in sightings:
    print(f"  - {s['species']:25s}  x{s['n']}  (best {s['best_conf']:.2f})")

## Step 10.3 — Format the digest

Build a Slack message body with the species list. Same Block Kit format as issue #7.

In [ ]:
def format_digest(sightings: list[dict]) -> dict:
    """Build the daily-digest Slack message."""
    total = sum(s["n"] for s in sightings)
    if not sightings:
        body_text = "Nothing showed up at the feeder yesterday. (Check the camera!)"
    else:
        lines = [f"  - {s['species']} (x{s['n']})" for s in sightings]
        body_text = f"Yesterday at the feeder ({total} sightings):\n" + "\n".join(lines)

    return {
        "blocks": [
            {"type": "header",
             "text": {"type": "plain_text", "text": "Bird Watcher — Daily Digest"}},
            {"type": "section",
             "text": {"type": "mrkdwn", "text": body_text}},
        ]
    }

import json, os
digest = format_digest(sightings)
print(json.dumps(digest, indent=2))

## Step 10.4 — Post (or dry-run)

Same `post_sighting` pattern from issue #7, but for digests.

In [ ]:
from dotenv import load_dotenv
import requests
import os

load_dotenv()

SLACK_WEBHOOK = os.environ.get("BIRD_WATCHER_WEBHOOK", "")

def post_digest() -> bool:
    msg = format_digest(yesterday_sightings())
    if not SLACK_WEBHOOK:
        print("[DRY-RUN digest]")
        print(msg["blocks"][1]["text"]["text"])
        return True
    r = requests.post(SLACK_WEBHOOK, json=msg, timeout=10)
    r.raise_for_status()
    print("[SENT digest]")
    return True

post_digest()

## Step 10.5 — Schedule it

The `schedule` library lets us say "run this function every day at 9am". Inside a notebook we can't actually wait 24 hours, so we'll schedule it every 30 seconds for demo purposes. Then you can change the interval for production.

In [ ]:
import schedule
import time

# In production: schedule.every().day.at("09:00").do(post_digest)
# For demo: every 30s so we can watch it run.
schedule.every(30).seconds.do(post_digest)

print("Scheduler started. Will post a digest every 30 seconds.")
print("Stop the cell when you've seen enough.\n")

try:
    for _ in range(3):  # 3 runs = 90s, then auto-stops
        schedule.run_pending()
        time.sleep(30)
except KeyboardInterrupt:
    print("Stopped by user")

print("\nDone. Bird watcher complete.")

## Step 10.6 — Production deployment

The notebook is for learning. For production, save the digest as a CLI command:

```bash
python -m bird_watcher digest
```

Then schedule it with cron on your server:

```
0 9 * * *  cd /path/to/bird-watcher && python -m bird_watcher digest
```

Every day at 9am, your server runs the digest, queries the db, and posts to Slack.

## Done!

All 10 boxes wired up. The bird watcher is complete. 🎉🐦

What you've built:

- ✅ Project skeleton (`python -m bird_watcher`)
- ✅ MJPEG snapshot grabber
- ✅ Polling loop with timestamps
- ✅ YOLO bird detection
- ✅ Bird species classifier
- ✅ SQLite sightings database
- ✅ Slack notifications on detection
- ✅ FastAPI web app
- ✅ Gallery view of all sightings
- ✅ Daily digest scheduler

What you learned along the way:

- How a streaming service reads frames
- What polling means and why we use it
- How object detection works (boxes + confidence)
- How chained models compose: detect → crop → classify
- Why we parameterize SQL queries (no injection)
- HTTP requests, webhooks, JSON messages
- Server-rendered HTML with templating
- Scheduling jobs with cron

That's not nothing. Real software engineers use these patterns every day. You're doing real engineering.

Go show your family. 🐦